# Potato CV – treino em GPU no Kaggle

Treina o detector de corpos estranhos (YOLO-cls + PaDiM) com a configuração maior de
`config.kaggle.yaml` e gera os modelos para usar no `run.py` da linha.

### Antes de rodar
1. **Dataset:** *Add Input → Upload* e envie `dataset_potatoes.zip` como um Kaggle Dataset
   (o Kaggle descompacta sozinho). Também funciona sem isso: o repositório já traz uma cópia.
2. **Settings → Accelerator:** `GPU T4 x2` ou `GPU P100`.
3. **Settings → Internet:** ligado (para clonar o código e baixar os pesos base do YOLO/ResNet).
4. *Run All*. O modelo final sai em `/kaggle/working/potato_models_gpu.zip` (aba **Output**).

> Com só ~757 imagens únicas, mais hardware não cria informação nova. Compare sempre pela
> **validação cruzada** (`RUN_CV = True`); o teste único tem poucas pedras/plantas e varia muito.

## 1. Opções

In [ ]:
# --- código ---
REPO_URL = "https://github.com/FabioAGC/Potato-Vision.git"
BRANCH   = "claude/festive-allen-tjezbb"   # troque para "main" depois do merge

# --- treino ---
CONFIG      = "config.kaggle.yaml"   # "config.yaml" = mesma configuração do CPU (para comparar)
RUN_CV      = True    # validação cruzada 5 partes (métrica confiável; é a parte mais demorada)
CV_FOLDS    = 5
RUN_UNKNOWN = True    # teste de objeto desconhecido (treina sem Pedra e depois sem Planta)

# Ajustes rápidos sem editar o YAML (None = usa o valor do arquivo). Ex.:
OVERRIDES = {
    # "classifier.model": "yolo11m-cls.pt",
    # "classifier.imgsz": 160,
    # "padim.backbone": "resnet18",
    # "padim.input_size": 48,
}

## 2. GPU

In [ ]:
import subprocess, torch
print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}:", torch.cuda.get_device_name(i))
else:
    print("ATENÇÃO: sem GPU. Ative em Settings → Accelerator (o treino vai rodar em CPU, bem mais lento).")

## 3. Código e dependências

In [ ]:
import os, shutil, subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
PROJ = WORK / "Potato-Vision"

if not (PROJ / "train.py").exists():
    # 1º: código anexado como Kaggle Dataset (útil sem internet)
    found = next((p.parent for p in Path("/kaggle/input").rglob("train.py")
                  if (p.parent / "potato_cv").is_dir()), None)
    if found:
        print("Código encontrado em", found)
        shutil.copytree(found, PROJ, ignore=shutil.ignore_patterns(".git"))
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(PROJ)], check=True)
os.chdir(PROJ)
print("Projeto em", Path.cwd())

In [ ]:
# PyTorch com CUDA já vem no Kaggle; instala só o resto (sem reinstalar o torch)
!pip install -q "ultralytics>=8.3" "onnx>=1.16" "onnxruntime>=1.18"
import ultralytics; print("Ultralytics", ultralytics.__version__)

## 4. Dataset

In [ ]:
import zipfile
IMG = {".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}
CLASSES = {"Good", "Damaged", "Plant", "Stone"}

def dataset_root(base: Path):
    """Pasta mais alta que contém pastas de classe (direto ou dentro de train/val/test)."""
    for d in sorted([base, *[p for p in base.rglob("*") if p.is_dir()]], key=lambda p: len(p.parts)):
        subs = {s.name for s in d.iterdir() if s.is_dir()}
        if subs & CLASSES or any({x.name for x in (d / s).iterdir() if x.is_dir()} & CLASSES for s in subs):
            return d
    return None

# zip enviado sem descompactar -> descompacta em /kaggle/working/dataset
for z in Path("/kaggle/input").rglob("*.zip"):
    with zipfile.ZipFile(z) as f:
        if any(n.lower().endswith(tuple(IMG)) for n in f.namelist()[:500]):
            dst = WORK / "dataset" / z.stem
            if not dst.exists():
                print("Descompactando", z); f.extractall(dst)

SOURCE = None
for base in [Path("/kaggle/input"), WORK / "dataset"]:
    if base.exists():
        for cand in base.iterdir():
            if cand.is_dir() and cand.resolve() != PROJ.resolve() and "Potato-Vision" not in str(cand):
                SOURCE = SOURCE or dataset_root(cand)
if SOURCE is None:
    SOURCE = PROJ / "dataset_potatoes" / "dataset_potatoes"
    print("Nenhum dataset em /kaggle/input – usando a cópia do repositório.")

from collections import Counter
cnt = Counter(p.parent.name for p in SOURCE.rglob("*") if p.suffix.lower() in IMG)
print("Dataset:", SOURCE)
print("Imagens por classe (antes de remover duplicatas):", dict(cnt), "| total", sum(cnt.values()))

## 5. Configuração usada

In [ ]:
import yaml
cfg = yaml.safe_load(open(CONFIG, encoding="utf-8"))
for k, v in OVERRIDES.items():
    if v is None:
        continue
    d = cfg
    *path, last = k.split(".")
    for p in path:
        d = d.setdefault(p, {})
    d[last] = v
RUN_CONFIG = "config.run.yaml"
yaml.safe_dump(cfg, open(RUN_CONFIG, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
print(yaml.safe_dump({k: cfg[k] for k in ("device", "classifier", "padim")}, allow_unicode=True, sort_keys=False))

# guarda as métricas do modelo de CPU (vindas do repositório) para comparar no fim
if Path("reports/metrics.json").exists() and not Path("reports/metrics_cpu.json").exists():
    shutil.copy("reports/metrics.json", "reports/metrics_cpu.json")

## 6. Treino

Tempo aproximado numa T4 com `config.kaggle.yaml`: modelo final ~20–40 min;
validação cruzada (5 partes) e teste de desconhecido somam algumas horas.
O limite de uma sessão no Kaggle é 12 h; para um teste rápido, use `RUN_CV = False`.

In [ ]:
args = ["python", "train.py", "--config", RUN_CONFIG, "--source", str(SOURCE)]
if RUN_CV:
    args += ["--cv", str(CV_FOLDS)]
if RUN_UNKNOWN:
    args += ["--unknown-test"]
print(" ".join(args))
p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
if p.wait() != 0:
    raise RuntimeError(f"train.py terminou com erro (código {p.returncode})")

## 7. Resultado

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path("reports/relatorio.md").read_text(encoding="utf-8")))

In [ ]:
import json
import pandas as pd

def resumo(path):
    m = json.loads(Path(path).read_text(encoding="utf-8"))
    out = {}
    for nome, bloco in (("Validação cruzada", m.get("validacao_cruzada")), ("Teste", m.get("teste"))):
        if not bloco:
            continue
        h, c = bloco["hibrido"], bloco["classificador"]
        out[nome] = {"Recall corpo estranho": h["recall_corpo_estranho"],
                     "Falso alarme batata": h["falso_alarme_batata"],
                     "Precisão": h["precisao"],
                     "Acurácia 4 classes": c["acuracia_4_classes"],
                     "AUROC PaDiM": bloco["padim"]["auroc"]}
    return out

linhas = {}
for rotulo, arq in (("CPU (repositório)", "reports/metrics_cpu.json"), ("GPU (este treino)", "reports/metrics.json")):
    if Path(arq).exists():
        for conj, vals in resumo(arq).items():
            linhas[(conj, rotulo)] = vals
df = pd.DataFrame(linhas).T
fmt = {c: "{:.1%}" for c in df.columns if c != "AUROC PaDiM"} | {"AUROC PaDiM": "{:.3f}"}
display(df.style.format(fmt, na_rep="–"))
print("Compare 'Validação cruzada' com 'Validação cruzada': é a única comparação justa.")

## 8. Exportar

In [ ]:
out = WORK / "potato_models_gpu.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    files = [*Path(cfg["classifier"]["weights"]).parent.glob("*"),
             Path("reports/relatorio.md"), Path("reports/metrics.json")]
    for f in files:
        if f.is_file():
            z.write(f, f.as_posix())
    # mesmo conteúdo do config.run.yaml, com o nome usado no README
    z.write(RUN_CONFIG, "config.kaggle.yaml")
print(out, f"{out.stat().st_size / 1e6:.1f} MB")
for n in zipfile.ZipFile(out).namelist():
    print("  ", n)

## Usar na máquina da linha

1. Baixe `potato_models_gpu.zip` na aba **Output** e descompacte na pasta do projeto
   (cria `models/gpu/`, `config.kaggle.yaml` e `reports/`).
2. Rode com a configuração nova:
   ```powershell
   & $py run.py --config config.kaggle.yaml
   ```
3. Os modelos maiores rodam em CPU, mas mais devagar. Veja `ms por recorte` no relatório
   e, se a tela ficar lenta, aumente `inference.every_n_frames` ou use `inference.engine: onnx`.

Só substitua os modelos atuais se o **recall** e o **falso alarme** da validação cruzada
forem melhores que os do modelo de CPU (tabela da seção 7).